# Frames benchmark and basline 

må ha rank_bm25 
https://www.geeksforgeeks.org/nlp/what-is-bm25-best-matching-25-algorithm/

## 1. Imports and more 

In [177]:
# used claude for debuggig and said this might be the issue with pyarrow and pandas. I don't know if this is the case but it seems to be a common issue. I will try to clear the pyarrow extension types and see if that fixes the issue.
import pyarrow as pa

for ext_type in ["pandas.period", "pandas.interval", "pandas.json"]:
    try:
        pa.unregister_extension_type(ext_type)
    except:
        pass

print("PyArrow extension types cleared")

PyArrow extension types cleared


In [178]:
# importing packages 
import re
import string
import os
from collections import Counter
from typing import List, Dict, Tuple
import ast
 
import numpy as np
import pandas as pd
#from tqdm.notebook import tqdm
from tqdm import tqdm

from rank_bm25 import BM25Okapi
from transformers import pipeline


In [179]:
#  path to the data
PASSAGE_CORPUS_PATH = "data/passage_corpus.csv"
GOLD_MAPPING_PATH = "data/gold_mapping.csv" 
OUTPUT_PATH = "results/baseline_results.csv"


# evaluation recall of the k retrieved passages
TOP_K_VALUES = [1, 3, 5, 10, 20]

# model for question answering
QA_MODEL = "distilbert-base-cased-distilled-squad"  # Changed from deepset/bert-base-cased-squad2 (this model doesn't work for some reason, maybe because it's a larger model and we don't have enough resources to run it? I will try a smaller model and see if it works)

# number of questions to evaluate on (set to None to evaluate on all questions) but we can set it to a smaller number for testing
MAX_QUESTIONS = None


os.makedirs("results", exist_ok=True)

print("Configuration set!")
print(f"  Passage corpus: {PASSAGE_CORPUS_PATH}")
print(f"  Gold mapping: {GOLD_MAPPING_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Top-k values: {TOP_K_VALUES}")
print(f"  Max questions: {MAX_QUESTIONS if MAX_QUESTIONS else 'All'}")

Configuration set!
  Passage corpus: data/passage_corpus.csv
  Gold mapping: data/gold_mapping.csv
  Output: results/baseline_results.csv
  Top-k values: [1, 3, 5, 10, 20]
  Max questions: All


## 2. convert gold mapping to csv 

## 3. loading the data

In [180]:
# Load passage corpus
print("Loading passage corpus...")
passages_df = pd.read_csv(PASSAGE_CORPUS_PATH)
print(f"  ✓ Loaded {len(passages_df):,} passages")

# Load gold mapping
print("\nLoading gold mapping...")
gold_df = pd.read_csv(GOLD_MAPPING_PATH)

# Parse gold_passage_ids from string back to list
gold_df['gold_passage_ids'] = gold_df['gold_passage_ids'].apply(ast.literal_eval)

print(f"  ✓ Loaded {len(gold_df):,} questions")

Loading passage corpus...
  ✓ Loaded 95,202 passages

Loading gold mapping...
  ✓ Loaded 824 questions


## 3. preprocessing text

https://www.geeksforgeeks.org/python/re-findall-in-python/

In [181]:
def tokenize(text: str) -> List[str]:
    # simple tokenization for BM25 

    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and split
    tokens = re.findall(r'\b\w+\b', text)
    return tokens

In [182]:
def normalize_answer(text: str) -> str:
    # normalize the answer for evaluation and this aslo follow with the tokenization for BM25

    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text) # removing a, an, the 

    def remove_punctuation(text):
        return ''.join(ch for ch in text if ch not in string.punctuation) # removing punctuation

    def remove_whitespace(text):
        return ' '.join(text.split())  # removing extra whitespace ( if there are multiple spaces, it will be reduced to a single space)
    
    return remove_whitespace(remove_punctuation(remove_articles(text.lower()))) 


# tests 
print(tokenize("The quick, brown fox!") == ["the", "quick", "brown", "fox"])
print(normalize_answer("The quick, brown fox!") == "quick brown fox")

# if these show true, then the functions are working correctly


True
True


## 4. Evaluation metric 


In [183]:
def compute_exact_match(prediction: str, ground_truth: str) -> float:
    # compute exact match score between the prediction and the ground truth answer

    return float(normalize_answer(prediction) == normalize_answer(ground_truth))

In [184]:
def compute_f1(prediction: str, ground_truth: str) -> float:
    # compute F1 score between the prediction and the ground truth answer
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()
    
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    
    common = Counter(pred_tokens) & Counter(gold_tokens) # count the number of common tokens between the prediction and the ground truth answer
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    
    return f1


In [185]:
def compute_recall_at_k(retrieved_passages: List[int], gold_ids: List[int], k: int) -> float:

    top_k_retrieved = set(retrieved_passages[:k]) # get the top k retrieved passages
    gold_ids_set = set(gold_ids) # convert gold ids to a set

    return float(len(top_k_retrieved & gold_ids_set) > 0) # compute the recall at k by checking if there is any intersection between the top k retrieved passages and the gold passage ids, and return 1.0 if there is at least one match, otherwise return 0.0
    


    """if len(gold_ids_set) == 0:
        return 0.0

    recall = len(top_k_retrieved & gold_ids_set) / len(gold_ids_set)
    return recall # compute the recall at k by calculating the intersection of the top k retrieved passages and the gold passage ids, and dividing by the number of gold passage ids

    # could use this and just remove the return statement above if we want to compute the recall as the proportion of gold passage ids that are retrieved in the top k, but for this task we only care about whether at least one gold passage id is retrieved in the top k, so we can return 1.0 if there is at least one match and 0.0 otherwise
"""



In [186]:
#testing 

# test cases for compute_exact_match 
print("\nTesting compute_exact_match():")
print("testing for match", {compute_exact_match('James II', 'James II')})
print("testing for match", {compute_exact_match('james ii', 'James II')})
print("testing for match", {compute_exact_match('james the only one', 'James II')})


# test cases for compute_f1
print("\nTesting compute_f1():")
print("testing for f1", {compute_f1('James II', 'James II')})
print("testing for f1", {compute_f1('James ii', 'James II')})
print("testing for f1", {compute_f1('james the only one', 'James II')})


# test cases for compute_recall_at_k
print("\nTesting compute_recall_at_k():")
print("testing for recall at k", {compute_recall_at_k([1, 2, 3], [2, 4, 5], 1)}) # should return 0.0 because the top 1 retrieved passage is 1, which is not in the gold ids [2, 4, 5]
print("testing for recall at k", {compute_recall_at_k([1, 2, 3], [2, 4, 5], 2)}) # should return 1.0 because the top 2 retrieved passages are [1, 2], and 2 is in the gold ids






Testing compute_exact_match():
testing for match {1.0}
testing for match {1.0}
testing for match {0.0}

Testing compute_f1():
testing for f1 {1.0}
testing for f1 {1.0}
testing for f1 {0.4}

Testing compute_recall_at_k():
testing for recall at k {0.0}
testing for recall at k {1.0}


## 5. BM25 retriver 

Very relevant use this 
https://stackoverflow.com/questions/61877065/implementation-of-okapi-bm25-in-python
https://github.com/crawlab-team/bm25/blob/main/bm25okapi.go
https://docs.langchain.com/oss/python/integrations/retrievers/bm25 # okapi for wikipedia 

Somewhat relevant: 
https://github.com/Rohith-2/bm25-fusion/blob/main/src/bm25_fusion/core.py

Not so relevant but take a look at this for furher reseach
https://github.com/xhluca/bm25s/tree/main/bm25s
https://developers.llamaindex.ai/python/framework/integrations/retrievers/bm25_retriever/   # from llama_index.retrievers.bm25 import BM25Retriever



"Okapi BM25" is the standard, variations exist to improve it:BM25F: A version that handles multiple fields (e.g., title, body, anchor text) with different importance.BM25+: Developed to fix a deficiency where long documents that match the query term are unfairly scored compared to shorter documents.rank_bm25 (Python): A popular Python library that includes implementations like BM25Okapi, BM25L, and BM25Plus

In [187]:
class BM25Retriever:
# BM25 retriever over a passage corpus.
# Uses the rank_bm25 library which implements BM25Okapi, the standard Okapi BM25 algorithm.     
    def __init__(self, passages_df: pd.DataFrame):
        # Initialize BM25 index from passage dataframe.
        print("Building BM25 index...")
        
        self.passages_df = passages_df.reset_index(drop=True)
        self.passage_ids = passages_df['passage_id'].tolist()
        
        # Tokenize all passages
        print("  Tokenizing passages...")
        self.tokenized_corpus = [
            tokenize(str(text)) for text in tqdm(passages_df['text'].tolist(), desc="  Tokenizing")
        ]
        
        # Build BM25 index
        print("  Building index...")
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        
        print(f"  ✓ Index built with {len(self.passage_ids):,} passages")
    
    def retrieve(self, query: str, top_k: int = 10) -> List[Tuple[int, float, str]]:
        # Retrieve top-k passages for a query.
        # Tokenize query
        tokenized_query = tokenize(query)
        
        # Get BM25 scores for all passages
        scores = self.bm25.get_scores(tokenized_query)
        
        # Get top-k indices (sorted by score descending)
        top_k_indices = np.argsort(scores)[::-1][:top_k]
        
        # Compile results
        results = []
        for idx in top_k_indices:
            passage_id = self.passage_ids[idx]
            score = scores[idx]
            text = self.passages_df.iloc[idx]['text']
            results.append((passage_id, score, text))
        
        return results

## 6. BERT Extractive QA

In [188]:
"""class BERTExtractiveQA:
    def __init__(self, model_name: str = QA_MODEL):
        self.qa_pipeline = pipeline("question-answering", model=model_name, device=1) 


    def extract_answer(self, question: str, context: str) -> Dict:
        try:
            result = self.qa_pipeline(question=question, context=context)
            return {
                'answer':result['answer'],
                'score':result['score'],
                'start':result['start'],
                'end':result['end']
            }
        except Exception as e:
            # Handle edge cases (empty context, etc.)
            return {
                'answer': '',
                'score': 0.0,
                'start': 0,
                'end': 0
            }"""

'class BERTExtractiveQA:\n    def __init__(self, model_name: str = QA_MODEL):\n        self.qa_pipeline = pipeline("question-answering", model=model_name, device=1) \n\n\n    def extract_answer(self, question: str, context: str) -> Dict:\n        try:\n            result = self.qa_pipeline(question=question, context=context)\n            return {\n                \'answer\':result[\'answer\'],\n                \'score\':result[\'score\'],\n                \'start\':result[\'start\'],\n                \'end\':result[\'end\']\n            }\n        except Exception as e:\n            # Handle edge cases (empty context, etc.)\n            return {\n                \'answer\': \'\',\n                \'score\': 0.0,\n                \'start\': 0,\n                \'end\': 0\n            }'

In [189]:
class BERTExtractiveQA:
    """
    BERT-based extractive QA model using DistilBERT.
    """
    
    def __init__(self, model_name: str = QA_MODEL):
        print(f"Loading QA model: {model_name}")
        print("  (This may take a minute on first run...)")
        
        from transformers import AutoModelForQuestionAnswering, AutoTokenizer
        import torch
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        
        print(f"  ✓ Model loaded on {self.device}")
    
    def extract_answer(self, question: str, context: str) -> Dict:
        """
        Extract answer span from context.
        """
        import torch
        
        try:
            # Tokenize
            inputs = self.tokenizer(
                question,
                context,
                return_tensors="pt",
                truncation=True,
                max_length=512
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            # Get predictions
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            # Get best start and end positions
            start_idx = torch.argmax(outputs.start_logits, dim=1).item()
            end_idx = torch.argmax(outputs.end_logits, dim=1).item()
            
            # Make sure end comes after start
            if end_idx < start_idx:
                end_idx = start_idx
            
            # Decode answer
            answer = self.tokenizer.decode(
                inputs["input_ids"][0][start_idx:end_idx + 1],
                skip_special_tokens=True
            )
            
            # Calculate confidence score
            start_probs = torch.softmax(outputs.start_logits, dim=0)
            end_probs = torch.softmax(outputs.end_logits, dim=0)
            score = (start_probs[0, start_idx].item() + end_probs[0, end_idx].item()) / 2
            
            return {
                'answer': answer.strip(),
                'score': score,
                'start': start_idx,
                'end': end_idx
            }
            
        except Exception as e:
            print(f"  Warning: QA error - {e}")
            return {
                'answer': '',
                'score': 0.0,
                'start': 0,
                'end': 0
            }

## 7. main eval 

In [190]:
def run_baseline_evaluation(
    passages_df: pd.DataFrame,
    gold_df: pd.DataFrame,
    top_k_values: List[int] = TOP_K_VALUES,
    max_questions: int = None
) -> pd.DataFrame:
    # Run the full baseline evaluation pipeline and return a DataFrame with results for each question.

    # could limit the number of questions for faster testing
    if max_questions:
        gold_df = gold_df.head(max_questions)
    
    print(f"\n{'='*60}")
    print("FRAMES Baseline Evaluation")
    print(f"{'='*60}")
    print(f"Passages: {len(passages_df):,}")
    print(f"Questions: {len(gold_df):,}")
    print(f"Top-k values: {top_k_values}")
    print(f"{'='*60}\n")
    
    # Initialize components
    retriever = BM25Retriever(passages_df)
    qa_model = BERTExtractiveQA()
    
    # Store results
    results_list = [] 
    max_k = max(top_k_values)
    
    print("\nRunning evaluation...")
    for idx, row in tqdm(gold_df.iterrows(), total=len(gold_df), desc="Evaluating"):
        question = row['prompt']
        gold_answer = str(row['answer'])
        gold_passage_ids = row['gold_passage_ids']
        
        # rertieve passages
        retrieved = retriever.retrieve(question, top_k=max_k)
        retrieved_ids = [pid for pid, score, text in retrieved]
        
        # compute Recall@k for each k
        recall_scores = {}
        for k in top_k_values:
            recall_scores[f'recall@{k}'] = compute_recall_at_k(
                retrieved_ids, gold_passage_ids, k
            )
        
        # Extract answer from top-1 passage
        if retrieved:
            top_passage_id, top_score, top_passage_text = retrieved[0]
            qa_result = qa_model.extract_answer(question, str(top_passage_text))
            predicted_answer = qa_result['answer']
            qa_confidence = qa_result['score']
        else:
            predicted_answer = ""
            qa_confidence = 0.0
            top_passage_id = None
            top_passage_text = ""
        
        # compute EM (Exact Match) and F1
        em = compute_exact_match(predicted_answer, gold_answer)
        f1 = compute_f1(predicted_answer, gold_answer)
        
        # Store result
        result = {
            'question_idx': row['question_idx'],
            'question': question,
            'gold_answer': gold_answer,
            'predicted_answer': predicted_answer,
            'em': em,
            'f1': f1,
            'qa_confidence': qa_confidence,
            'top_passage_id': top_passage_id,
            'n_gold_passages': len(gold_passage_ids),
            **recall_scores
        }
        results_list.append(result)
    
    results_df = pd.DataFrame(results_list)
    return results_df

In [191]:
def print_summary(results_df: pd.DataFrame, top_k_values: List[int]):
   # print evaluation summary statistics
    print(f"\n{'='*60}")
    print("EVALUATION RESULTS")
    print(f"{'='*60}")
    
    print("\n--- Retrieval Performance (BM25) ---")
    for k in top_k_values:
        col = f'recall@{k}'
        mean_recall = results_df[col].mean()
        print(f"  Recall@{k}: {mean_recall:.4f} ({mean_recall*100:.1f}%)")
    
    print("\n--- QA Performance (BERT) ---")
    print(f"  Exact Match (EM): {results_df['em'].mean():.4f} ({results_df['em'].mean()*100:.1f}%)")
    print(f"  F1 Score:         {results_df['f1'].mean():.4f} ({results_df['f1'].mean()*100:.1f}%)")
    
    print("\n--- Error Analysis ---")
    # Retrieval failures: gold passage not in top-k
    retrieval_failures = (results_df['recall@10'] == 0).sum()
    print(f"  Retrieval failures (gold not in top-10): {retrieval_failures} ({retrieval_failures/len(results_df)*100:.1f}%)")
    
    # Cases where retrieval succeeded but QA failed
    retrieval_success_qa_fail = ((results_df['recall@10'] == 1) & (results_df['em'] == 0)).sum()
    print(f"  Retrieval OK but QA failed: {retrieval_success_qa_fail} ({retrieval_success_qa_fail/len(results_df)*100:.1f}%)")
    
    print(f"\n{'='*60}\n")

## 9. running evaluation 

In [192]:
#MAX_QUESTIONS = 10  # Test with 10 first

results_df = run_baseline_evaluation(
    passages_df=passages_df,
    gold_df=gold_df,
    top_k_values=TOP_K_VALUES,
    max_questions=MAX_QUESTIONS
)



FRAMES Baseline Evaluation
Passages: 95,202
Questions: 824
Top-k values: [1, 3, 5, 10, 20]

Building BM25 index...
  Tokenizing passages...


  Tokenizing: 100%|██████████| 95202/95202 [00:00<00:00, 121816.81it/s]


  Building index...
  ✓ Index built with 95,202 passages
Loading QA model: distilbert-base-cased-distilled-squad
  (This may take a minute on first run...)


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 17952.21it/s]


  ✓ Model loaded on cpu

Running evaluation...


Evaluating: 100%|██████████| 824/824 [06:20<00:00,  2.16it/s]


In [193]:
print_summary(results_df, TOP_K_VALUES)


EVALUATION RESULTS

--- Retrieval Performance (BM25) ---
  Recall@1: 0.7039 (70.4%)
  Recall@3: 0.8507 (85.1%)
  Recall@5: 0.8956 (89.6%)
  Recall@10: 0.9260 (92.6%)
  Recall@20: 0.9539 (95.4%)

--- QA Performance (BERT) ---
  Exact Match (EM): 0.0170 (1.7%)
  F1 Score:         0.0502 (5.0%)

--- Error Analysis ---
  Retrieval failures (gold not in top-10): 61 (7.4%)
  Retrieval OK but QA failed: 749 (90.9%)




In [194]:
# Look at what the model is predicting
print("=== Sample Predictions ===\n")
for i in range(min(5, len(results_df))):
    row = results_df.iloc[i]
    print(f"Q: {row['question'][:80]}...")
    print(f"Gold Answer:      '{row['gold_answer']}'")
    print(f"Predicted Answer: '{row['predicted_answer']}'")
    print(f"Recall@10: {row['recall@10']}, EM: {row['em']}, F1: {row['f1']:.2f}")
    print("-" * 50)

=== Sample Predictions ===

Q: If my future wife has the same first name as the 15th first lady of the United S...
Gold Answer:      'Jane Ballou'
Predicted Answer: 'Seti II'
Recall@10: 0.0, EM: 0.0, F1: 0.00
--------------------------------------------------
Q: Imagine there is a building called Bronte tower whose height in feet is the same...
Gold Answer:      '37th'
Predicted Answer: '650'
Recall@10: 1.0, EM: 0.0, F1: 0.00
--------------------------------------------------
Q: How many years earlier would Punxsutawney Phil have to be canonically alive to h...
Gold Answer:      '87'
Predicted Answer: '2024'
Recall@10: 1.0, EM: 0.0, F1: 0.00
--------------------------------------------------
Q: As of August 1, 2024, which country were holders of the FIFA World Cup the last ...
Gold Answer:      'France'
Predicted Answer: 'As of August 1, 2024, which country were holders of the FIFA World Cup the last time the UEFA Champions League was won by a club from London'
Recall@10: 1.0, EM: 0.0,

# Debugging and checking aswers

In [195]:
# Debug: Check what the QA model is actually returning
qa_model = BERTExtractiveQA()

# Test with a simple example first
test_question = "What is the capital of France?"
test_context = "Paris is the capital of France. It is a beautiful city."

result = qa_model.extract_answer(test_question, test_context)
print(f"Test Question: {test_question}")
print(f"Test Context: {test_context}")
print(f"Result: {result}")

Loading QA model: distilbert-base-cased-distilled-squad
  (This may take a minute on first run...)


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 21027.18it/s]


  ✓ Model loaded on cpu
Test Question: What is the capital of France?
Test Context: Paris is the capital of France. It is a beautiful city.
Result: {'answer': 'Paris', 'score': 1.0, 'start': 9, 'end': 9}


In [196]:
# Quick test before running full evaluation
qa_model = BERTExtractiveQA()

question = "What is the capital of France?"
context = "Paris is the capital of France. It is a beautiful city."

result = qa_model.extract_answer(question, context)
print(f"Test: {result}")
# Should print: {'answer': 'Paris', 'score': ..., 'start': ..., 'end': ...}

Loading QA model: distilbert-base-cased-distilled-squad
  (This may take a minute on first run...)


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 17631.12it/s]


  ✓ Model loaded on cpu
Test: {'answer': 'Paris', 'score': 1.0, 'start': 9, 'end': 9}


In [197]:
MAX_QUESTIONS = 10  # Test with 10 first

results_df = run_baseline_evaluation(
    passages_df=passages_df,
    gold_df=gold_df,
    top_k_values=TOP_K_VALUES,
    max_questions=MAX_QUESTIONS
)

print_summary(results_df, TOP_K_VALUES)


FRAMES Baseline Evaluation
Passages: 95,202
Questions: 10
Top-k values: [1, 3, 5, 10, 20]

Building BM25 index...
  Tokenizing passages...


  Tokenizing: 100%|██████████| 95202/95202 [00:01<00:00, 93324.78it/s] 


  Building index...
  ✓ Index built with 95,202 passages
Loading QA model: distilbert-base-cased-distilled-squad
  (This may take a minute on first run...)


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 16121.60it/s]


  ✓ Model loaded on cpu

Running evaluation...


Evaluating: 100%|██████████| 10/10 [00:05<00:00,  1.81it/s]


EVALUATION RESULTS

--- Retrieval Performance (BM25) ---
  Recall@1: 0.8000 (80.0%)
  Recall@3: 0.8000 (80.0%)
  Recall@5: 0.9000 (90.0%)
  Recall@10: 0.9000 (90.0%)
  Recall@20: 0.9000 (90.0%)

--- QA Performance (BERT) ---
  Exact Match (EM): 0.0000 (0.0%)
  F1 Score:         0.0049 (0.5%)

--- Error Analysis ---
  Retrieval failures (gold not in top-10): 1 (10.0%)
  Retrieval OK but QA failed: 9 (90.0%)




In [198]:
# Look at what the model is predicting
print("=== Sample Predictions ===\n")
for i in range(min(10, len(results_df))):
    row = results_df.iloc[i]
    print(f"Q: {row['question'][:80]}...")
    print(f"Gold Answer:      '{row['gold_answer']}'")
    print(f"Predicted Answer: '{row['predicted_answer']}'")
    print(f"Recall@10: {row['recall@10']}, EM: {row['em']}, F1: {row['f1']:.2f}")
    print("-" * 50)

=== Sample Predictions ===

Q: If my future wife has the same first name as the 15th first lady of the United S...
Gold Answer:      'Jane Ballou'
Predicted Answer: 'Seti II'
Recall@10: 0.0, EM: 0.0, F1: 0.00
--------------------------------------------------
Q: Imagine there is a building called Bronte tower whose height in feet is the same...
Gold Answer:      '37th'
Predicted Answer: '650'
Recall@10: 1.0, EM: 0.0, F1: 0.00
--------------------------------------------------
Q: How many years earlier would Punxsutawney Phil have to be canonically alive to h...
Gold Answer:      '87'
Predicted Answer: '2024'
Recall@10: 1.0, EM: 0.0, F1: 0.00
--------------------------------------------------
Q: As of August 1, 2024, which country were holders of the FIFA World Cup the last ...
Gold Answer:      'France'
Predicted Answer: 'As of August 1, 2024, which country were holders of the FIFA World Cup the last time the UEFA Champions League was won by a club from London'
Recall@10: 1.0, EM: 0.0,